In [14]:
import sys
import pandas as pd
from sqlalchemy import text

#защита окружения
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()

def q(sql: str, **params) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()

#задаем окно фактов
WINDOW = ("2026-01-01", "2026-07-01")

#выгружаем клиентов
clients = q("""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
""")

#выгружаем подписки
subscriptions = q("""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at
    from prime.subscriptions
""")

#выгружаем успешные платежи
payments = q("""
    select payment_id, subscription_id, paid_at, amount, status
    from prime.payments
    where paid_at >= :a and paid_at < :b
      and status = 'success'
""", a=WINDOW[0], b=WINDOW[1])

#выгружаем тикеты поддержки
support_tickets = q("""
    select ticket_id, client_id, category, created_at, resolved_at
    from prime.support_tickets
    where created_at >= :a and created_at < :b
""", a=WINDOW[0], b=WINDOW[1])

for name, frame in [("clients", clients), ("subscriptions", subscriptions), ("payments", payments), ("support_tickets", support_tickets)]:
    print(f"{name:<16} {len(frame):>9} строк")

print("\nКатегории в тикетах:", support_tickets['category'].unique())

clients             500000 строк
subscriptions       333905 строк
payments           1045826 строк
support_tickets      55084 строк

Категории в тикетах: <ArrowStringArray>
['кэшбэк', 'доступ к сервису', 'отмена подписки', 'шеринг', 'оплата',
 'прочее']
Length: 6, dtype: str
